In [4]:
# =========================
# train_model.py
# Projet MBA1 Finance Digitale – ISM Dakar
# Modèle final : Régression Logistique
# =========================

import joblib
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score
)

# =========================
# 1. CHARGEMENT DES DONNÉES
# =========================
df = pd.read_csv("dataset_scoring_credit_900k.csv")

# =========================
# 2. VARIABLES SÉLECTIONNÉES (TOP 10)
# =========================
FEATURES = [
    "RATIO_ENDETTEMENT",
    "ANCIENNETE_CLIENT_MOIS",
    "ANCIENNETE_EMPLOI_MOIS",
    "AGE",
    "GENRE",
    "TYPE_LOGEMENT",
    "TYPE_CARTE",
    "SITUATION_MATRIMONIALE",
    "FLAG_LISTE_NOIRE",
    "REVENU_DISPONIBLE_FCFA"
]

TARGET = "DEFAUT"

X = df[FEATURES]
y = df[TARGET]

numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = X.select_dtypes(include=["object"]).columns

# =========================
# 3. PREPROCESSING (CLASSE)
# =========================
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

# =========================
# 4. TRAIN / TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# =========================
# 5. PIPELINE RÉGRESSION LOGISTIQUE
# =========================
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        random_state=42,
        class_weight="balanced",
        max_iter=1000
    ))
])

# =========================
# 6. ENTRAÎNEMENT
# =========================
pipeline.fit(X_train, y_train)

# =========================
# 7. ÉVALUATION DES PERFORMANCES
# =========================
# Probabilités
y_proba = pipeline.predict_proba(X_test)[:, 1]

# Prédictions (seuil par défaut = 0.5)
y_pred = pipeline.predict(X_test)

auc = roc_auc_score(y_test, y_proba)
recall = recall_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("\n📊 Performances du modèle (jeu de test)")
print(f"AUC ROC            : {auc:.4f}")
print(f"Recall (défaut)    : {recall:.4f}")
print(f"Precision (défaut) : {precision:.4f}")
print(f"F1-score           : {f1:.4f}")

# =========================
# 8. SAUVEGARDE DU MODÈLE
# =========================
joblib.dump(pipeline, "credit_scoring_model.pkl")
print("\n✅ Modèle sauvegardé : credit_scoring_model.pkl")



📊 Performances du modèle (jeu de test)
AUC ROC            : 0.9990
Recall (défaut)    : 0.9903
Precision (défaut) : 0.8362
F1-score           : 0.9067

✅ Modèle sauvegardé : credit_scoring_model.pkl


In [5]:
import numpy as np

# Probabilités prédites par le modèle
y_proba = pipeline.predict_proba(X_test)[:, 1]

# Prédictions automatiques de scikit-learn
y_pred_predict = pipeline.predict(X_test)

# Prédictions manuelles avec seuil explicite à 0.5
SEUIL = 0.5
y_pred_manual = (y_proba >= SEUIL).astype(int)

# Vérification
difference = np.sum(y_pred_predict != y_pred_manual)

print(f"Seuil utilisé : {SEUIL}")
print(f"Nombre de différences entre predict() et seuil manuel : {difference}")


Seuil utilisé : 0.5
Nombre de différences entre predict() et seuil manuel : 0
